# 02 Denoising Autoencoder Baseline

Image-to-image reconstruction template based on the denoising exercise.

In [ ]:
# Colab setup if needed
# from google.colab import drive
# drive.mount("/content/drive")
# !pip install -q -e .

In [ ]:
import numpy as np
from tensorflow.keras.datasets import mnist

from cnugs_ml.data import prepare_images, add_gaussian_noise, describe_array, set_global_seed
from cnugs_ml.models import build_denoising_autoencoder
from cnugs_ml.plots import plot_image_grid, plot_loss_history, plot_prediction_triplets
from cnugs_ml.train import make_early_stopping

set_global_seed(42)

## Load clean images and make noisy inputs

In [ ]:
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

clean_train = prepare_images(train_images, add_channel=True)
clean_test = prepare_images(test_images, add_channel=True)

noise_sigma = 0.3
noisy_train = add_gaussian_noise(clean_train, sigma=noise_sigma, seed=42)
noisy_test = add_gaussian_noise(clean_test, sigma=noise_sigma, seed=123)

describe_array("clean_train", clean_train)
describe_array("noisy_train", noisy_train)
plot_image_grid(noisy_train, train_labels, n=6, rows=2)

## Build denoiser

In [ ]:
denoiser = build_denoising_autoencoder(
    input_shape=clean_train.shape[1:],
    filters=(32, 32),
    learning_rate=1e-3,
    output_activation="sigmoid",
)
denoiser.summary()

## Train image-to-image model

Notice the target is the clean image, not a class label.

In [ ]:
early_stopping = make_early_stopping(monitor="val_loss", patience=5)
history = denoiser.fit(
    noisy_train, clean_train,
    validation_split=0.2,
    epochs=30,
    batch_size=128,
    callbacks=[early_stopping],
)
plot_loss_history(history)

## Visual check

In [ ]:
pred = denoiser.predict(noisy_test[:5])
plot_prediction_triplets(noisy_test[:5], pred, clean_test[:5], n=5)

## Evaluate

In [ ]:
test_loss = denoiser.evaluate(noisy_test, clean_test)
print(f"test MSE = {test_loss:.6f}")